# Healthcare Admissions — Data Cleaning Pipeline

Cleans and validates a synthetic healthcare dataset (generated with [Synthea](https://synthetichealth.github.io/synthea/)) for use in a Power BI star-schema admissions dashboard.

**Pipeline steps per table:** load → profile → clean → validate → export.

**Design decisions**
- Dimension tables (`patients`, `providers`, `organizations`, `payers`) are cleaned and exported at **full scope** — no date filtering. These describe entities, not events, so trimming them by date would silently drop valid dimension members that a fact table might still reference.
- Fact / event tables (`encounters`, `conditions`, `procedures`, `claims`, `claims_transactions`) are cleaned at full scope first, then a **single, consistent** analysis window (`ANALYSIS_START`–`ANALYSIS_END`) is applied identically across all of them in the final section — so every fact table reflects the same reporting period.
- Encounter dates in this dataset are real (unshifted) Synthea timestamps, so calendar-year filtering and trending is valid here (unlike a MIMIC-style de-identified date-shifted dataset).

## Setup & Configuration

In [ ]:
import pandas as pd
from pathlib import Path
import logging

# Configure professional logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)-8s | %(message)s"
)
logger = logging.getLogger(__name__)

# Project paths
PROJECT_ROOT = Path("..")
RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)

# Temporal configuration: all fact tables windowed to this period
ANALYSIS_START = pd.Timestamp("2020-01-01", tz="UTC")
ANALYSIS_END = pd.Timestamp("2022-12-31 23:59:59", tz="UTC")
REFERENCE_DATE = pd.Timestamp("2022-12-31")

logger.info("\n" + "="*70)
logger.info("HEALTHCARE DATA CLEANING PIPELINE")
logger.info("="*70)
logger.info(f"Analysis window: {ANALYSIS_START} to {ANALYSIS_END}")
logger.info(f"Reference date for age calculation: {REFERENCE_DATE}\n")

## Helper Functions

In [ ]:
def profile_dataframe(df: pd.DataFrame, name: str) -> None:
    """Print a one-line data quality snapshot."""
    missing_count = df.isna().sum().sum()
    duplicate_count = df.duplicated().sum()
    logger.info(
        f"[{name:30}] rows={len(df):>9,}  cols={df.shape[1]:>2}  "
        f"duplicates={duplicate_count:>6,}  missing={missing_count:>9,}"
    )

def load_raw_csv(filename: str) -> pd.DataFrame:
    """Load a raw Synthea CSV and log its profile."""
    df = pd.read_csv(RAW_DATA / filename)
    profile_dataframe(df, filename.replace(".csv", ""))
    return df

def export_clean_csv(df: pd.DataFrame, filename: str) -> None:
    """Export a cleaned table after profiling."""
    profile_dataframe(df, filename.replace(".csv", ""))
    df.to_csv(PROCESSED_DATA / filename, index=False)
    logger.info(f"✓ Exported: {filename}")

def apply_temporal_window(df: pd.DataFrame, date_column: str, table_name: str) -> pd.DataFrame:
    """Filter a fact table to the shared analysis window."""
    before_count = len(df)
    windowed = df[(df[date_column] >= ANALYSIS_START) & (df[date_column] <= ANALYSIS_END)].copy()
    after_count = len(windowed)
    pct_retained = (after_count / before_count * 100) if before_count > 0 else 0
    logger.info(f"[{table_name:20}] {before_count:>9,} → {after_count:>9,} ({pct_retained:.1f}% retained)")
    return windowed

## 1. Patients (Dimension)

In [ ]:
patients_raw = load_raw_csv("patients.csv")

keep_cols = [
    "Id", "BIRTHDATE", "DEATHDATE", "MARITAL", "RACE", "ETHNICITY", "GENDER",
    "CITY", "STATE", "COUNTY", "FIPS", "ZIP", "LAT", "LON",
    "HEALTHCARE_EXPENSES", "HEALTHCARE_COVERAGE", "INCOME",
]
patients_clean = patients_raw[keep_cols].copy()

patients_clean["BIRTHDATE"] = pd.to_datetime(patients_clean["BIRTHDATE"], errors="coerce")
patients_clean["DEATHDATE"] = pd.to_datetime(patients_clean["DEATHDATE"], errors="coerce")
patients_clean["MARITAL"] = patients_clean["MARITAL"].fillna("Unknown")
patients_clean["IS_DECEASED"] = patients_clean["DEATHDATE"].notna()

age_days = (REFERENCE_DATE - patients_clean["BIRTHDATE"]).dt.days
age_years = (age_days / 365.25).astype(int)
patients_clean["AGE"] = age_years.where(age_years >= 0, pd.NA)

patients_clean["AGE_GROUP"] = pd.cut(
    patients_clean["AGE"],
    bins=[0, 17, 34, 49, 64, 79, 120],
    labels=["0-17", "18-34", "35-49", "50-64", "65-79", "80+"],
    include_lowest=True,
)

export_clean_csv(patients_clean, "patients_clean.csv")
assert patients_clean["Id"].is_unique, "ERROR: Duplicate patient IDs"
assert (patients_clean["AGE"].dropna() >= 0).all(), "ERROR: Negative ages detected"
logger.info(f"   Patients with AGE assigned: {patients_clean['AGE'].notna().sum():,}\n")

## 2. Providers (Dimension)

In [ ]:
providers_raw = load_raw_csv("providers.csv")
providers_clean = providers_raw.copy()

assert providers_clean["Id"].is_unique, "ERROR: Duplicate provider IDs"
assert ((providers_clean["LAT"].between(-90, 90)) & (providers_clean["LON"].between(-180, 180))).all(), "ERROR: Invalid coordinates"

export_clean_csv(providers_clean, "providers_clean.csv")
logger.info(f"   Providers in dataset: {len(providers_clean):,}\n")

## 3. Organizations (Dimension)

In [ ]:
organizations_raw = load_raw_csv("organizations.csv")
organizations_clean = organizations_raw.copy()

assert organizations_clean["Id"].is_unique, "ERROR: Duplicate organization IDs"
assert (organizations_clean["REVENUE"] >= 0).all(), "ERROR: Negative revenue values"
assert (organizations_clean["UTILIZATION"] >= 0).all(), "ERROR: Negative utilization values"

export_clean_csv(organizations_clean, "organizations_clean.csv")
logger.info(f"   Organizations: {len(organizations_clean):,}\n")

## 4. Payers (Dimension)

In [ ]:
payers_raw = load_raw_csv("payers.csv")
payers_clean = payers_raw.copy()

assert payers_clean["Id"].is_unique, "ERROR: Duplicate payer IDs"
assert (payers_clean["AMOUNT_COVERED"] >= 0).all(), "ERROR: Negative covered amounts"
assert (payers_clean["AMOUNT_UNCOVERED"] >= 0).all(), "ERROR: Negative uncovered amounts"

export_clean_csv(payers_clean, "payers_clean.csv")
logger.info(f"   Payers: {len(payers_clean):,}\n")

## 5. Encounters (Fact)

Cleaned at full scope with derived columns (duration, category, patient responsibility).
Temporal windowing applied later in Section 9 alongside all other fact tables.

In [ ]:
encounters_raw = load_raw_csv("encounters.csv")
encounters_clean = encounters_raw.copy()

encounters_clean["START"] = pd.to_datetime(encounters_clean["START"], errors="coerce")
encounters_clean["STOP"] = pd.to_datetime(encounters_clean["STOP"], errors="coerce")

encounters_clean["ENCOUNTER_DURATION_HOURS"] = (
    (encounters_clean["STOP"] - encounters_clean["START"]).dt.total_seconds() / 3600
).round(2)

encounters_clean["ENCOUNTER_YEAR"] = encounters_clean["START"].dt.year
encounters_clean["ENCOUNTER_MONTH"] = encounters_clean["START"].dt.month
encounters_clean["ENCOUNTER_MONTH_NAME"] = encounters_clean["START"].dt.month_name()

encounter_category_map = {
    "ambulatory": "Outpatient", "outpatient": "Outpatient",
    "wellness": "Preventive", "urgentcare": "Urgent Care", "emergency": "Emergency",
    "inpatient": "Inpatient", "home": "Home Care", "snf": "Skilled Nursing",
    "virtual": "Telehealth", "hospice": "Hospice",
}
encounters_clean["ENCOUNTER_CATEGORY"] = encounters_clean["ENCOUNTERCLASS"].map(encounter_category_map)

encounters_clean["PATIENT_RESPONSIBILITY"] = (
    encounters_clean["TOTAL_CLAIM_COST"] - encounters_clean["PAYER_COVERAGE"]
).round(2)
encounters_clean["PAYER_COVERAGE_RATE"] = (
    encounters_clean["PAYER_COVERAGE"] / encounters_clean["TOTAL_CLAIM_COST"]
).round(4)

export_clean_csv(encounters_clean, "encounters_clean.csv")
assert encounters_clean["Id"].is_unique, "ERROR: Duplicate encounter IDs"
assert (encounters_clean["STOP"] >= encounters_clean["START"]).all(), "ERROR: STOP before START"
assert encounters_clean["ENCOUNTER_CATEGORY"].notna().all(), "ERROR: Unmapped ENCOUNTERCLASS"
logger.info(f"   Encounter categories: {encounters_clean['ENCOUNTER_CATEGORY'].nunique()}")
logger.info(f"   Avg payer coverage: {encounters_clean['PAYER_COVERAGE_RATE'].mean():.1%}\n")

## 6. Conditions (Fact)

Categorized into 18 clinical buckets via keyword matching.

In [ ]:
conditions_raw = load_raw_csv("conditions.csv")
conditions_clean = conditions_raw.copy()

conditions_clean["START"] = pd.to_datetime(conditions_clean["START"], errors="coerce")
conditions_clean["STOP"] = pd.to_datetime(conditions_clean["STOP"], errors="coerce")

conditions_clean["CONDITION_NAME"] = (
    conditions_clean["DESCRIPTION"].str.replace(r"\s*\([^)]*\)$", "", regex=True).str.strip()
)

category_keywords = {
    "Dental": ["gingiv", "dental", "tooth", "teeth", "caries", "molar"],
    "Cardiovascular": ["myocardial", "heart failure", "heart disease", "hypertension", "coronary"],
    "Respiratory": ["sinusitis", "bronchitis", "pneumonia", "asthma", "emphysema"],
    "Metabolic / Endocrine": ["diabetes", "obesity", "hyperlipidemia", "metabolic syndrome"],
    "Musculoskeletal / Injury": ["fracture", "sprain", "injury", "laceration", "burn"],
    "Genitourinary": ["kidney", "renal", "cystitis", "urinary", "bladder"],
    "Neurological": ["seizure", "migraine", "epilepsy", "stroke", "dementia"],
    "Women's Health": ["pregnan", "miscarriage", "prenatal", "postpartum"],
    "Infectious Disease": ["infection", "viral", "bacterial", "coronavirus", "sepsis"],
    "Mental / Behavioral": ["stress", "anxiety", "depress", "alcoholism", "drug abuse"],
    "Oncology": ["cancer", "carcinoma", "leukemia", "lymphoma", "melanoma"],
    "Hematologic": ["anemia", "coagulation", "neutropenia"],
    "Gastrointestinal": ["appendic", "vomiting", "nausea", "gastro", "diarrhea"],
    "Dermatological": ["dermatitis", "eczema", "rash"],
    "Symptoms": ["fever", "cough", "headache", "fatigue", "dyspnea"],
    "Social / SDOH": ["unemployed", "homeless", "social isolation", "housing"],
    "Care / Administrative": ["medication review"],
    "Care / End-of-Life": ["hospice"],
}

def categorize_condition(name: str) -> str:
    lowered = name.lower()
    for category, keywords in category_keywords.items():
        if any(kw in lowered for kw in keywords):
            return category
    return "Unclassified"

conditions_clean["CLINICAL_CATEGORY"] = conditions_clean["CONDITION_NAME"].apply(categorize_condition)

export_clean_csv(conditions_clean, "conditions_clean.csv")
unclassified_pct = (conditions_clean["CLINICAL_CATEGORY"] == "Unclassified").mean()
logger.info(f"   Unclassified rate: {unclassified_pct:.2%}\n")

## 7. Procedures (Fact)

In [ ]:
procedures_raw = load_raw_csv("procedures.csv")
procedures_clean = procedures_raw.copy()

procedures_clean["START"] = pd.to_datetime(procedures_clean["START"], errors="coerce")
procedures_clean["STOP"] = pd.to_datetime(procedures_clean["STOP"], errors="coerce")

procedures_clean["PROCEDURE_DURATION_HOURS"] = (
    (procedures_clean["STOP"] - procedures_clean["START"]).dt.total_seconds() / 3600
).round(2)

before = len(procedures_clean)
procedures_clean = procedures_clean[procedures_clean["PROCEDURE_DURATION_HOURS"] >= 0].copy()
if before > len(procedures_clean):
    logger.warning(f"   ⚠ Dropped {before - len(procedures_clean)} row(s) with negative duration")

export_clean_csv(procedures_clean, "procedures_clean.csv")
assert (procedures_clean["PROCEDURE_DURATION_HOURS"] >= 0).all(), "ERROR: Negative durations"
logger.info("")

## 8. Claims & Claims Transactions (Fact)

In [ ]:
claims_raw = load_raw_csv("claims.csv")
claims_clean = claims_raw.copy()

for col in ["CURRENTILLNESSDATE", "SERVICEDATE", "LASTBILLEDDATE1", "LASTBILLEDDATE2", "LASTBILLEDDATEP"]:
    claims_clean[col] = pd.to_datetime(claims_clean[col], errors="coerce")

for col in ["STATUS1", "STATUS2", "STATUSP"]:
    claims_clean[col] = claims_clean[col].str.strip().str.upper()

claims_clean["TOTAL_OUTSTANDING"] = (
    claims_clean["OUTSTANDING1"].fillna(0) + claims_clean["OUTSTANDING2"].fillna(0) + claims_clean["OUTSTANDINGP"].fillna(0)
).round(2)
claims_clean["HAS_OUTSTANDING_BALANCE"] = claims_clean["TOTAL_OUTSTANDING"] > 0

export_clean_csv(claims_clean, "claims_clean.csv")
assert claims_clean["Id"].is_unique, "ERROR: Duplicate claim IDs"
outstanding_pct = claims_clean["HAS_OUTSTANDING_BALANCE"].mean()
logger.info(f"   Claims with outstanding balance: {outstanding_pct:.1%}\n")

In [ ]:
claims_txn_raw = load_raw_csv("claims_transactions.csv")
claims_txn_clean = claims_txn_raw.copy()

claims_txn_clean["FROMDATE"] = pd.to_datetime(claims_txn_clean["FROMDATE"], errors="coerce")
claims_txn_clean["TODATE"] = pd.to_datetime(claims_txn_clean["TODATE"], errors="coerce")
claims_txn_clean["TYPE"] = claims_txn_clean["TYPE"].str.strip().str.upper()

for col in ["AMOUNT", "PAYMENTS", "ADJUSTMENTS", "TRANSFERS", "OUTSTANDING"]:
    claims_txn_clean[col] = pd.to_numeric(claims_txn_clean[col], errors="coerce").fillna(0).round(2)

export_clean_csv(claims_txn_clean, "claims_transactions_clean.csv")
assert claims_txn_clean["ID"].is_unique, "ERROR: Duplicate transaction IDs"
logger.info("")

## 9. Temporal Windowing — Apply 2020-2022 Analysis Window

Critical: All fact tables are windowed to the same period for consistent Power BI reporting.

In [ ]:
logger.info("\n" + "="*70)
logger.info("APPLYING TEMPORAL WINDOW: 2020-01-01 to 2022-12-31")
logger.info("="*70 + "\n")

encounters_2020_2022 = apply_temporal_window(encounters_clean, "START", "encounters")
conditions_2020_2022 = apply_temporal_window(conditions_clean, "START", "conditions")
procedures_2020_2022 = apply_temporal_window(procedures_clean, "START", "procedures")
claims_2020_2022 = apply_temporal_window(claims_clean, "SERVICEDATE", "claims")
claims_txn_2020_2022 = apply_temporal_window(claims_txn_clean, "FROMDATE", "claims_transactions")

encounters_2020_2022.to_csv(PROCESSED_DATA / "encounters_2020_2022.csv", index=False)
conditions_2020_2022.to_csv(PROCESSED_DATA / "conditions_2020_2022.csv", index=False)
procedures_2020_2022.to_csv(PROCESSED_DATA / "procedures_2020_2022.csv", index=False)
claims_2020_2022.to_csv(PROCESSED_DATA / "claims_2020_2022.csv", index=False)
claims_txn_2020_2022.to_csv(PROCESSED_DATA / "claims_transactions_2020_2022.csv", index=False)

logger.info("\n✓ All windowed tables exported to data/processed/")

## 10. Pipeline Summary

In [ ]:
logger.info("\n" + "="*70)
logger.info("PIPELINE COMPLETE — SUMMARY")
logger.info("="*70 + "\n")

summary = pd.DataFrame([
    {"table": "patients_clean", "rows": f"{len(patients_clean):,}", "scope": "full"},
    {"table": "providers_clean", "rows": f"{len(providers_clean):,}", "scope": "full"},
    {"table": "organizations_clean", "rows": f"{len(organizations_clean):,}", "scope": "full"},
    {"table": "payers_clean", "rows": f"{len(payers_clean):,}", "scope": "full"},
    {"table": "encounters_clean", "rows": f"{len(encounters_clean):,}", "scope": "full"},
    {"table": "encounters_2020_2022", "rows": f"{len(encounters_2020_2022):,}", "scope": "windowed"},
    {"table": "conditions_clean", "rows": f"{len(conditions_clean):,}", "scope": "full"},
    {"table": "conditions_2020_2022", "rows": f"{len(conditions_2020_2022):,}", "scope": "windowed"},
    {"table": "procedures_clean", "rows": f"{len(procedures_clean):,}", "scope": "full"},
    {"table": "procedures_2020_2022", "rows": f"{len(procedures_2020_2022):,}", "scope": "windowed"},
    {"table": "claims_clean", "rows": f"{len(claims_clean):,}", "scope": "full"},
    {"table": "claims_2020_2022", "rows": f"{len(claims_2020_2022):,}", "scope": "windowed"},
    {"table": "claims_transactions_clean", "rows": f"{len(claims_txn_clean):,}", "scope": "full"},
    {"table": "claims_transactions_2020_2022", "rows": f"{len(claims_txn_2020_2022):,}", "scope": "windowed"},
])

print(summary.to_string(index=False))
logger.info(f"\n✓ All cleaned data exported to: {PROCESSED_DATA}")